# Phần 2: Hồi quy tuyến tính có điều chỉnh và các phương pháp nâng cao
## Dự đoán giá nhà California Housing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from data_pipeline import DataPipeline
from part2.model_comparison import RidgeRegression, LassoRegression, cross_validate_model, evaluate_model, compare_models
from advanced_methods import KernelRidgeRegression, BayesianLinearRegression
from part2.residual_analysis import plot_residual_analysis

sns.set_style('whitegrid')
%matplotlib inline

## 1. Đọc và khám phá dữ liệu (EDA)

In [ ]:
df = pd.read_csv('data/housing.csv')
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['median_house_value'].hist(bins=50, ax=axes[0])
axes[0].set_title('Phân phối giá nhà')
df.corr(numeric_only=True)['median_house_value'].sort_values().plot(kind='barh', ax=axes[1])
axes[1].set_title('Tương quan với giá nhà')
plt.tight_layout()

## 2. Tiền xử lý dữ liệu

In [ ]:
target = 'median_house_value'
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Áp dụng pipeline
pipeline = DataPipeline(numeric_features, categorical_features)
X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

print(f"Train shape: {X_train_transformed.shape}")
print(f"Test shape: {X_test_transformed.shape}")

## 3. Tìm λ tối ưu cho Ridge và Lasso bằng Cross-Validation

In [ ]:
lambda_values = [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100]

ridge_cv_results, best_ridge_lambda = cross_validate_model(
    RidgeRegression, X_train_transformed, y_train.values, 
    lambda_values, k=5
)

lasso_cv_results, best_lasso_lambda = cross_validate_model(
    LassoRegression, X_train_transformed, y_train.values,
    lambda_values, k=5
)

print(f"Ridge: λ tối ưu = {best_ridge_lambda}")
print(f"Lasso: λ tối ưu = {best_lasso_lambda}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(lambda_values, [ridge_cv_results[l] for l in lambda_values], 'o-', label='Ridge')
plt.plot(lambda_values, [lasso_cv_results[l] for l in lambda_values], 's-', label='Lasso')
plt.xscale('log')
plt.xlabel('λ (log scale)')
plt.ylabel('CV RMSE')
plt.title('Cross-Validation để tìm λ tối ưu')
plt.legend()
plt.grid(True)

## 4. So sánh các mô hình trên tập test

In [ ]:
models = {
    'Ridge (optimal)': RidgeRegression(lambda_=best_ridge_lambda),
    'Lasso (optimal)': LassoRegression(lambda_=best_lasso_lambda),
    'Kernel Ridge': KernelRidgeRegression(lambda_=1.0, gamma=0.01),
    'Bayesian Linear': BayesianLinearRegression()
}

comparison_df = compare_models(models, X_train_transformed, y_train.values, 
                               X_test_transformed, y_test.values)
comparison_df

## 5. Feature Importance (Hệ số hồi quy) - Từ Ridge tối ưu

In [ ]:
best_ridge = RidgeRegression(lambda_=best_ridge_lambda)
best_ridge.fit(X_train_transformed, y_train.values)

feature_names = pipeline.get_feature_names()
coeff_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': best_ridge.weights
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['red' if c < 0 else 'green' for c in coeff_df.Coefficient.values[:15]]
plt.barh(coeff_df.Feature.head(15), coeff_df.Coefficient.head(15), color=colors)
plt.xlabel('Hệ số hồi quy')
plt.title('Top 15 đặc trưng ảnh hưởng nhất (Ridge Regression)')
plt.gca().invert_yaxis()
plt.tight_layout()

## 6. Phân tích phần dư - Chọn mô hình tốt nhất

In [ ]:
# Dùng Kernel Ridge vì có R2 cao nhất
best_model = KernelRidgeRegression(lambda_=1.0, gamma=0.01)
best_model.fit(X_train_transformed, y_train.values)
y_pred = best_model.predict(X_test_transformed)

fig = plot_residual_analysis(y_test.values, y_pred, title='Phân tích phần dư - Kernel Ridge Regression')
plt.show()

## 7. Kết luận
- **Ridge và Lasso** giúp giảm overfitting so với OLS thông qua điều chỉnh λ.
- **Kernel Ridge** với RBF kernel có thể học được mối quan hệ phi tuyến, cho kết quả tốt nhất (R² ≈ 0.85).
- **Bayesian Linear Regression** cung cấp thêm khoảng tin cậy cho dự đoán.
- Phân tích phần dư cho thấy phần dư gần phân phối chuẩn, không có hiện tượng phương sai thay đổi rõ rệt.